# Transcriptomics

**Course:** Computational Methods / Computational Biology  
**Institution:** Universidad Nacional Autónoma de México (UNAM)

---

## Author Information
* **Name:** Héctor Alessandro López Hernández
* **GitHub Username**: `@alssandrohrnandz`
* **Repository:** `Metodos_Biologia_Computacional_UNAM`
* **Date Creation:** 2026-09-27

## Objectives
1. Understanding the integration of Python with international biological databases (NCBI) using **Bio.Entrez**.
2. Retrieving and processing files in industry-standard formats (**GenBank** y **FASTA**) using **Bio.SeqIO**.
3. Applying *in silico* molecular operations (transcriptions, translation, GC content calculation and restriction motif search).
4. Structuring and generating a reproducible bioinformatics report.

## Enviroments & Dependencies
List the primary packages and libraries required to run this notebook:
* `biopython...`
* `Entrez`
* `Medline`
* `SeqIO`
* `gc_fraction`

## 📚 References
* **Title:** *Biopython: freeware Python tools for computational biology and bioinformatics*  
* **Authors:** Cock, P. J. A., et al. (2009). *Bioinformatics*, 25(11), 1422–1423.  
* **PMID:** `19304878`

## Módulo 1: Conexión y Consultas a NCBI via `Bio.Entrez`

El conjunto de herramientas **E-Utilities** de NCBI permite realizar búsquedas programáticas en bases de datos como PubMed, Nucleotide y Protein. Para acceder a estos servicios con Biopython, es obligatorio proporcionar un correo electrónico mediante `Entrez.email` para recibir avisos del servidor en caso de tráfico excesivo.

In [1]:
# !pip install biopython
from Bio import Entrez, Medline

# 1. Configuración obligatoria del correo electrónico
Entrez.email = "estudiante@universidad.edu"

# 2. Búsqueda de literatura en PubMed sobre el gen HBB
search_term = "hemoglobin beta sickle cell"
handle = Entrez.esearch(db="pubmed", term=search_term, retmax=5)
results = Entrez.read(handle)
handle.close()

pmid_list = results["IdList"]
print(f"PubMed IDs encontrados para '{search_term}': {pmid_list}")

# 3. Recuperación de metadatos bibliográficos completos
fetch_handle = Entrez.efetch(db="pubmed", id=pmid_list, rettype="medline", retmode="text")
records = list(Medline.parse(fetch_handle))
fetch_handle.close()

print("\n--- Resumen de Publicaciones Recuperadas ---")
for i, record in enumerate(records, 1):
    title = record.get("TI", "Sin título")
    authors = ", ".join(record.get("AU", ["Autores no listados"]))
    journal = record.get("TA", "Revista no disponible")
    pub_date = record.get("DP", "Fecha N/A")
    print(f"{i}. {title}")
    print(f"   Autores: {authors}")
    print(f"   Revista: {journal} ({pub_date})\n")

PubMed IDs encontrados para 'hemoglobin beta sickle cell': ['42793718', '42792753', '42787623', '42756451', '42755327']

--- Resumen de Publicaciones Recuperadas ---
1. Clinical Implications of Sickle Cell Thalassemia on Pregnancy Outcomes: A Systematic Review.
   Autores: Gerede A, Stavros S, Potiris A, Oikonomou E, Christodoulaki C, Vogiatzoglou AL, Chatzakis C, Domali E, Koutlaki N, Eleftheriadis M
   Revista: Diagnostics (Basel) (2026 Sep 11)

2. Mitochondrial Damage in Hemoglobinopathies: The Role of Cell-Free mtDNA as a Potential Biomarker.
   Autores: Neijmann ST, Gedikbasi A, Tuncer FN, Dogan SA, Tatonyan S, Balci MC, Bekmez B, Karaman V, Gokcay GF, Karakas Z
   Revista: Biomedicines (2026 Sep 7)

3. Clinical translation of CRISPR-Cas9 therapeutics in cancer and inherited genetic disorders.
   Autores: Bashir K, Vasudev P, Chhetri V, Dubhashi DP, Teli YG, Biswas R, Ramaiah S, Anbarasu A
   Revista: Front Genome Ed (2026)

4. The Influence of Fetal Hemoglobin on Hematological Pa

## Módulo 2: Recuperación e Inspección de Registros Moleculares (`Bio.SeqIO`)

Utilizaremos la clave de acceso de GenBank **`NM_000518.5`**, correspondiente al ARNm del gen de la Hemoglobina Subunidad Beta (*HBB*) en *Homo sapiens*, para descargar e inspeccionar sus anotaciones biológicas directamente desde la base de datos Nucleotide de NCBI.

In [3]:
from Bio import SeqIO

# 1. Descarga del registro en formato GenBank (.gb)
acc_id = "NM_000518.5"
handle = Entrez.efetch(db="nucleotide", id=acc_id, rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()

print(f"ID del Registro: {record.id}")
print(f"Descripción: {record.description}")
print(f"Organismo: {record.annotations.get('organism', 'Desconocido')}")
print(f"Longitud total del transcripto: {len(record.seq)} pb")

# 2. Inspección de anotaciones moleculares (Features: CDS, UTRs, Exones)
print("\n--- Características Anotadas (Features) ---")
for feature in record.features:
    if feature.type in ["CDS", "5'UTR", "3'UTR"]:
        start = feature.location.start
        end = feature.location.end
        strand = "+" if feature.location.strand == 1 else "-"
        print(f"[Tipo: {feature.type:<6}] Posición: {start}:{end} (Hebra {strand})")
        
        if feature.type == "CDS":
            prot_seq = feature.qualifiers.get("translation", ["N/A"])[0]
            print(f"   Proteína anotada en NCBI ({len(prot_seq)} aa):")
            print(f"   {prot_seq}")

ID del Registro: NM_000518.5
Descripción: Homo sapiens hemoglobin subunit beta (HBB), mRNA
Organismo: Homo sapiens
Longitud total del transcripto: 628 pb

--- Características Anotadas (Features) ---
[Tipo: CDS   ] Posición: 50:494 (Hebra +)
   Proteína anotada en NCBI (147 aa):
   MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH


## Módulo 3: Análisis y Procesamiento Molecular de Secuencias

En esta sección realizaremos análisis moleculares *in silico* con las secuencias obtenidas:
- Cálculo del contenido GC (proporción de Guanina y Citosina).
- Transcripción y traducción de la secuencia nucleotídica.
- Búsqueda de motivos de restricción (e.g., diana para *EcoRI*: `GAATTC`).

In [4]:
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

dna_seq = record.seq

# 1. Cálculo de contenido GC%
gc_percent = gc_fraction(dna_seq) * 100
print(f"Contenido GC de {record.id}: {gc_percent:.2f}%")

# 2. Transcripción in silico (ADN -> ARNm)
mrna_seq = dna_seq.transcribe()
print(f"ARNm (primeros 40 nt): {mrna_seq[:40]}...")

# 3. Búsqueda de la región codificante (CDS) y traducción
for feature in record.features:
    if feature.type == "CDS":
        cds_seq = feature.extract(dna_seq)
        protein_seq = cds_seq.translate(to_stop=True)
        
        print(f"\nSecuencia CDS ({len(cds_seq)} pb): {cds_seq[:30]}...")
        print(f"Proteína traducida in silico ({len(protein_seq)} aa):")
        print(protein_seq)
        break

# 4. Mapeo de sitio de restricción EcoRI (GAATTC)
restriction_site = "GAATTC"
position = dna_seq.find(restriction_site)

if position != -1:
    print(f"\nSitio de restricción EcoRI ({restriction_site}) detectado en la posición: {position}")
else:
    print(f"\nEl sitio de restricción EcoRI ({restriction_site}) NO se encuentra en la secuencia.")

Contenido GC de NM_000518.5: 51.27%
ARNm (primeros 40 nt): ACAUUUGCUUCUGACACAACUGUGUUCACUAGCAACCUCA...

Secuencia CDS (444 pb): ATGGTGCATCTGACTCCTGAGGAGAAGTCT...
Proteína traducida in silico (147 aa):
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH

Sitio de restricción EcoRI (GAATTC) detectado en la posición: 413


## Módulo 4: Evaluación y Guía para el Reporte de Práctica

### Instrucciones para la elaboración del Reporte de Práctica
Cada estudiante o pareja de trabajo deberá elegir un gen de interés biológico (por ejemplo: `TP53`, `INS`, `EGFR`, `BRCA1`) y replicar las actividades analíticas vistas en los módulos 1 a 3.

---

### Plantilla para la entrega del Reporte

#### **1. Datos Generales**
- **Nombre del Estudiante:** 
- **Fecha:** 
- **Gen / Proteína Analizado:** 
- **Especie de Estudio:** 

#### **2. Búsqueda Bibliográfica Automatizada**
- Término de búsqueda en PubMed utilizado:
- Metadatos del artículo principal encontrado:
  - **PMID:** 
  - **Título:** 
  - **Primer Autor:** 
  - **Revista y Año:** 

#### **3. Caracterización de la Secuencia Nucleotídica**
- **Accession Number (NCBI Nucleotide):** 
- **Longitud de la secuencia (pb):** 
- **Porcentaje GC (%):** 
- **Coordenadas de la CDS (Region Codificante):** 

#### **4. Análisis Biológico e In Silico**
- Primers 15 aminoácidos de la secuencia traducida:
- Presencia de sitios de corte por enzima de restricción (*EcoRI* o la enzima de su elección):
- **Pregunta de Discusión:** ¿Por qué la traducción directa de la secuencia completa de ARNm (comenzando desde el nucleótido 1) difiere de la traducción del fragmento CDS anotado? Explique el rol biológico del 5' UTR.